# Minimal placeholder pipeline for bond valuation with embedded options

This notebook reads a CSV placeholder from the `data/` folder, constructs a tiny bond specification, and values the embedded option with the repository's Hull-White multi-curve tree.


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys
from pathlib import Path
import os

sys.path.insert(0, str(Path.cwd().parents[1]))
from src.curve import YieldCurve
from src.multi_hw_tree import BondSpec, CouponDef, ExerciseSpec, MultiCurveHWTree
from src.daycount import DayCount

In [ ]:
root = Path.cwd().resolve().parent.parent
BOND_DATA_PATH = root / 'datasets' /'raw'/ 'bonds_placeholder.csv'
CURVE_DATA_PATH = root / 'datasets' / 'raw' / 'Histocopy_FI_ZYC_VND_GD2_family.xlsx'
START_DATE = pd.to_datetime('2024-06-24')
END_DATE = pd.to_datetime('2026-07-30')
REPORT_DATE = pd.to_datetime('2026-06-03')
DISC_CONVENTION = 'actactisda'
COUP_CONVENTION = 'act365'

In [14]:
#Define the groups and initialize dictionaries to hold dataframes and discount curves
groups = ['LB_G1', 'LB_G2', 'LB_G3', 'LB_G4', 'NBFI', 'FB']
dfs = {}
disc_curves = {}

daycount = DayCount.get(DISC_CONVENTION)

for group in groups:
    df = pd.read_excel(
        CURVE_DATA_PATH,
        sheet_name=f'FI ZYC VND_GD2_{group}',
        index_col=0,
    )

    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    date_range = pd.date_range(start = START_DATE, end = END_DATE)
    df = df.reindex(date_range)

    dfs[group] = df

    tenor_labels = [col.split('_')[-1] for col in df.columns]
    maturity_dates = []

    for tenor in tenor_labels:
        if tenor.endswith('Y'):
            n = int(tenor[:-1])
            maturity_dates.append(REPORT_DATE + pd.DateOffset(years=n))
        elif tenor.endswith('M'):
            n = int(tenor[:-1])
            maturity_dates.append(REPORT_DATE + pd.DateOffset(months=n))
        else:
            raise ValueError(f'Unsupported tenor: {tenor}')

    start = np.array([REPORT_DATE] * len(maturity_dates), dtype='datetime64[D]')
    end = np.array(maturity_dates, dtype='datetime64[D]')

    maturities = daycount.yearfrac(start, end)
    zero_rates = df.loc[REPORT_DATE].to_numpy(dtype=float)

    disc_curves[group] = YieldCurve.from_zero_rates(
        maturities=maturities,
        zero_rates=zero_rates,
    )

LB_G1 0.92674947819476
LB_G2 0.9299987843371915
LB_G3 0.9237193382162319
LB_G4 0.9237193382162319
NBFI 0.9114769255450744
FB 0.9173731176556816


In [ ]:
#Import bond_df and define the MultiCurveHWTree for each bond
bond_df = pd.read_csv(BOND_DATA_PATH)

#Define coupon_rate of each bond in float rate case and fix rate case
for idx, row in bond_df.iterrows():
    if row['coupon_type'] == 'float':
        bond_df.at[idx, 'coupon_rate'] = YieldCurve.from_zero_rates(
            maturities=np.array([0.5, 1.0, 2.0, 3.0, 4.0, 5.0]),
            zero_rates=np.array([0.02, 0.025, 0.03, 0.035, 0.04, 0.045])
        ).discount(1.0)  # Example: using discount factor at 1 year



for idx, row in bond_df.iterrows():
    group = row['group']
    if group not in disc_curves:
        raise ValueError(f"Group {group} not found in discount curves.")

    disc_curve = disc_curves[group]

    bond_spec = BondSpec(
        issue_date=pd.to_datetime(row['issue_date']),
        maturity_date=pd.to_datetime(row['maturity_date']),
        coupon_rate=row['coupon_rate'],
        coupon_freq=row['coupon_freq'],
        daycount=COUP_CONVENTION,
        face_value=row['face_value'],
    )

    exercise_spec = ExerciseSpec(
        exercise_type=row['exercise_type'],
        exercise_dates=[pd.to_datetime(d) for d in row['exercise_dates'].split(';')],
    )

    coupon_def = CouponDef(
        coupon_rate=row['coupon_rate'],
        coupon_freq=row['coupon_freq'],
        daycount=COUP_CONVENTION,
    )

    hw_tree = MultiCurveHWTree(
        bond_spec=bond_spec,
        exercise_spec=exercise_spec,
        coupon_def=coupon_def,
        disc_curve=disc_curve,
    )


In [ ]:
groups = ['LB_G1', 'LB_G2', 'LB_G3', 'LB_G4', 'NBFI', 'FB']
dfs = {}
disc_curves = {}

daycount = DayCount.get(DISC_CONVENTION)

for group in groups:
    df = pd.read_excel(
        CURVE_DATA_PATH,
        sheet_name=f'FI ZYC VND_GD2_{group}',
        index_col=0,
    )

    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    date_range = pd.date_range(start=START_DATE, end=END_DATE)
    df = df.reindex(date_range)
    dfs[group] = df

    tenor_labels = [col.split('_')[-1] for col in df.columns]
    maturity_dates = []

    for tenor in tenor_labels:
        if tenor.endswith('Y'):
            n = int(tenor[:-1])
            maturity_dates.append(REPORT_DATE + pd.DateOffset(years=n))
        elif tenor.endswith('M'):
            n = int(tenor[:-1])
            maturity_dates.append(REPORT_DATE + pd.DateOffset(months=n))
        else:
            raise ValueError(f'Unsupported tenor: {tenor}')

    start = np.array([REPORT_DATE] * len(maturity_dates), dtype='datetime64[D]')
    end = np.array(maturity_dates, dtype='datetime64[D]')
    maturities = daycount.yearfrac(start, end)

    valid_idx = df.index[df.index <= REPORT_DATE]
    report_idx = valid_idx.max()
    zero_rates = df.loc[report_idx].to_numpy(dtype=float)

    disc_curves[group] = YieldCurve.from_zero_rates(
        maturities=maturities,
        zero_rates=zero_rates,
    )


In [ ]:
bond_df = pd.read_csv(BOND_DATA_PATH)
display(bond_df)


bond_group_map = {
    'BOND_EUR_01': 'LB_G1',
    'BOND_BERM_02': 'FB',
}

# coupon cố định / thả nổi
coupon_map_fixed = {
    1: CouponDef(accrual=0.25, fixed_rate=0.04),
    2: CouponDef(accrual=0.25, fixed_rate=0.04),
    3: CouponDef(accrual=0.25, fixed_rate=0.04),
    4: CouponDef(accrual=0.25, fixed_rate=0.04),
}

coupon_map_float = {
    1: CouponDef(accrual=0.25, fixed_rate=None, margin=0.0025, ref_tenor=0.25),
    2: CouponDef(accrual=0.25, fixed_rate=None, margin=0.0025, ref_tenor=0.25),
    3: CouponDef(accrual=0.25, fixed_rate=None, margin=0.0025, ref_tenor=0.50),
    4: CouponDef(accrual=0.25, fixed_rate=None, margin=0.0025, ref_tenor=0.50),
}

results = []

for _, row in bond_df.iterrows():
    bond_id = row['bond_id']
    group = bond_group_map[bond_id]

    tree = MultiCurveHWTree(
        a_r=0.03,
        sigma_r=0.015,
        a_L=0.02,
        sigma_L=0.018,
        rho=0.20,
        disc_curve=curves[group],
        ref_curve=ref_curves[group],
        dt=1.0,
        n_steps=4,
    )

    # bond type theo style
    if row['style'] == 'European':
        coupon_map = coupon_map_fixed
        bond = BondSpec(
            face=float(row['face']),
            maturity_step=int(row['maturity_step']),
            coupons=coupon_map,
        )
        exercise = ExerciseSpec(call={int(row['exercise_step']): float(row['strike'])})

        full_price = tree.price(bond, exercise)
        straight_price = tree.price(bond)

    elif row['style'] == 'Bermudan':
        coupon_map = coupon_map_float
        bond = BondSpec(
            face=float(row['face']),
            maturity_step=int(row['maturity_step']),
            coupons=coupon_map,
        )
        exercise = ExerciseSpec(put={2: 99.0, 4: 99.0})

        full_price = tree.price(bond, exercise)
        straight_price = tree.price(bond)

    else:
        raise ValueError(f"Unsupported style: {row['style']}")

    results.append({
        'bond_id': bond_id,
        'group': group,
        'style': row['style'],
        'full_price': full_price,
        'straight_price': straight_price,
    })

pd.DataFrame(results)

In [ ]:
coupon_map_fixed = {
    1: CouponDef(accrual=0.25, fixed_rate=0.04),
    2: CouponDef(accrual=0.25, fixed_rate=0.04),
    3: CouponDef(accrual=0.25, fixed_rate=0.04),
    4: CouponDef(accrual=0.25, fixed_rate=0.04),
}

In [ ]:
coupon_map_float = {
    1: CouponDef(
        accrual=0.25,
        fixed_rate=None,
        margin=0.0025,
        ref_tenor=0.25,   # 3M reference
    ),
    2: CouponDef(
        accrual=0.25,
        fixed_rate=None,
        margin=0.0025,
        ref_tenor=0.25,
    ),
    3: CouponDef(
        accrual=0.25,
        fixed_rate=None,
        margin=0.0025,
        ref_tenor=0.50,   # 6M reference
    ),
    4: CouponDef(
        accrual=0.25,
        fixed_rate=None,
        margin=0.0025,
        ref_tenor=0.50,
    ),
}

In [ ]:
results = []

for group in groups:
    tree = MultiCurveHWTree(
        a_r=0.03,
        sigma_r=0.015,
        a_L=0.02,
        sigma_L=0.018,
        rho=0.20,
        disc_curve=curves[group],
        ref_curve=ref_curves[group],
        dt=1.0,
        n_steps=4,
    )

    # bond cố định
    bond_european = BondSpec(
        face=100.0,
        maturity_step=4,
        coupons=coupon_map_fixed,
    )
    exercise_european = ExerciseSpec(call={4: 101.0})

    # bond thả nổi
    bond_bermudan_float = BondSpec(
        face=100.0,
        maturity_step=4,
        coupons=coupon_map_float,
    )
    exercise_bermudan = ExerciseSpec(put={2: 99.0, 4: 99.0})

    results.append({
        'group': group,
        'european_full_price': tree.price(bond_european, exercise_european),
        'european_straight_price': tree.price(bond_european),
        'bermudan_full_price': tree.price(bond_bermudan_float, exercise_bermudan),
        'bermudan_straight_price': tree.price(bond_bermudan_float),
    })

pd.DataFrame(results)

In [ ]:

placeholder = pd.read_csv(BOND_DATA_PATH)
display(placeholder)
tree = MultiCurveHWTree(
    a_r=0.03,
    sigma_r=0.015,
    a_L=0.02,
    sigma_L=0.018,
    rho=0.20,
    disc_curve=curve,
    ref_curve=curve,
    dt=1.0,
    n_steps=4,
)

coupon_map = {
    1: CouponDef(accrual=0.25, fixed_rate=0.04),
    2: CouponDef(accrual=0.25, fixed_rate=0.04),
    3: CouponDef(accrual=0.25, fixed_rate=0.04),
    4: CouponDef(accrual=0.25, fixed_rate=0.04),
}

bond_european = BondSpec(face=100.0, maturity_step=4, coupons=coupon_map)
exercise_european = ExerciseSpec(call={4: 101.0})

bond_bermudan = BondSpec(face=100.0, maturity_step=4, coupons=coupon_map)
exercise_bermudan = ExerciseSpec(put={2: 99.0, 4: 99.0})

results = {
    'bond_type': ['European call', 'Bermudan put'],
    'full_price': [
        tree.price(bond_european, exercise_european),
        tree.price(bond_bermudan, exercise_bermudan),
    ],
    'straight_price': [
        tree.price(bond_european),
        tree.price(bond_bermudan),
    ],
}

pd.DataFrame(results)
print(results)

In [6]:


groups = ['LB_G1', 'LB_G2', 'LB_G3', 'LB_G4', 'NBFI', 'FB']
dfs = {}
curves = {}

daycount = DayCount.get(DISC_CONVENTION) 
tenor_map = {
    '3M': 0.25,
    '6M': 0.50,
    '9M': 0.75,
    '1Y': 1.00,
    '15M': 1.25,
    '18M': 1.50,
    '21M': 1.75,
    '2Y': 2.00,
    '27M': 2.25,
    '30M': 2.50,
    '33M': 2.75,
    '3Y': 3.00,
    '5Y': 5.00,
}

for group in groups:
    df = pd.read_excel(
        CURVE_DATA_PATH,
        sheet_name=f'FI ZYC VND_GD2_{group}',
        index_col=0,
    )

    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    date_range = pd.date_range(start=START_DATE, end=END_DATE)
    df = df.reindex(date_range)

    dfs[group] = df

    # maturities được suy từ tên cột
    maturities = np.array(
        [tenor_map[col.split('_')[-1]] for col in df.columns],
        dtype=float,
    )

    # zero rates lấy theo ngày báo cáo
    zero_rates = df.loc[REPORT_DATE].to_numpy(dtype=float)

    curves[group] = YieldCurve.from_zero_rates(
        maturities=maturities,
        zero_rates=zero_rates,
    )
    print (curves[group].discount)


YieldCurve(maturities=array([0.25, 0.5 , 0.75, 1.  , 1.25, 1.5 , 1.75, 2.  , 2.25, 2.5 , 2.75,
       3.  , 5.  ]), zero_rates=array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]))
YieldCurve(maturities=array([0.25, 0.5 , 0.75, 1.  , 1.25, 1.5 , 1.75, 2.  , 2.25, 2.5 , 2.75,
       3.  , 5.  ]), zero_rates=array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]))
YieldCurve(maturities=array([0.25, 0.5 , 0.75, 1.  , 1.25, 1.5 , 1.75, 2.  , 2.25, 2.5 , 2.75,
       3.  , 5.  ]), zero_rates=array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]))
YieldCurve(maturities=array([0.25, 0.5 , 0.75, 1.  , 1.25, 1.5 , 1.75, 2.  , 2.25, 2.5 , 2.75,
       3.  , 5.  ]), zero_rates=array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan]))
YieldCurve(maturities=array([0.25, 0.5 , 0.75, 1.  , 1.25, 1.5 , 1.75, 2.  , 2.25, 2.5 , 2.75,
       3.  , 5.  ]), zero_rates=array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, na

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

from src.daycount import DayCount
from src.curve import YieldCurve

root = Path.cwd().resolve().parent.parent
groups = ['LB_G1', 'LB_G2', 'LB_G3', 'LB_G4', 'NBFI', 'FB']

dfs = {}
curves = {}

daycount = DayCount.get("act365")   # hoặc "act360", "actactisda"

for group in groups:
    df = pd.read_csv(root / 'datasets' / 'raw' / f'bonds_{group}.csv')
    dfs[group] = df

    # Giả sử file có 2 cột ngày:
    # settlement_date, maturity_date, zero_rate
    start = df['settlement_date'].to_numpy(dtype='datetime64[D]')
    end = df['maturity_date'].to_numpy(dtype='datetime64[D]')

    # Tính year fraction theo daycount convention
    maturities = daycount.yearfrac(start, end)

    # Lấy zero rate làm input cho curve
    zero_rates = df['zero_rate'].to_numpy(dtype=float)

    curves[group] = YieldCurve.from_zero_rates(
        maturities=maturities,
        zero_rates=zero_rates,
    )

# Ví dụ kiểm tra curve của group
display(curves['LB_G1'].discount(1.0))

FileNotFoundError: [Errno 2] No such file or directory: 'E:\\VCB\\202607\\vcb-snh\\datasets\\raw\\bonds_LB_G1.csv'

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import sys

sys.path.insert(0, str(Path.cwd().parents[1]))

from src.curve import YieldCurve
from src.daycount import DayCount
from src.multi_hw_tree import BondSpec, CouponDef, ExerciseSpec, MultiCurveHWTree

# -----------------------------
# 1) Thiết lập đường dẫn & tham số
# -----------------------------
root = Path.cwd().resolve().parent.parent

BOND_DATA_PATH = root / "data" / "raw" / "bonds_placeholder.csv"
CURVE_DATA_PATH = root / "data" / "raw" / "Histocopy_FI_ZYC_VND_GD2_family.xlsx"

START_DATE = pd.to_datetime("2024-06-24")
END_DATE = pd.to_datetime("2026-07-30")
REPORT_DATE = pd.to_datetime("2026-06-03")

DISC_CONVENTION = "actactisda"
COUP_CONVENTION = "act365"

# Nếu bond_df có cột group thì dùng cột đó.
# Nếu không có, map tay theo bond_id.
bond_group_map = {
    "BOND_EUR_01": "LB_G1",
    "BOND_BERM_02": "FB",
}

# Nếu bond_df có cột ref_curve thì map theo giá trị đó
# Ví dụ: "LB_G1" hoặc "FB" hoặc "group_1"...
ref_curve_map = {
    "BOND_EUR_01": "LB_G1",
    "BOND_BERM_02": "FB",
}

# -----------------------------
# 2) Dựng curve theo từng group
# -----------------------------
groups = ["LB_G1", "LB_G2", "LB_G3", "LB_G4", "NBFI", "FB"]

dfs = {}
curves = {}
ref_curves = {}

daycount = DayCount.get(DISC_CONVENTION)

for group in groups:
    df = pd.read_excel(
        CURVE_DATA_PATH,
        sheet_name=f"FI ZYC VND_GD2_{group}",
        index_col=0,
    )

    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    date_range = pd.date_range(start=START_DATE, end=END_DATE)
    df = df.reindex(date_range)
    dfs[group] = df

    # Tên cột dạng: <group>_3M, <group>_6M, <group>_1Y, ...
    tenor_labels = [col.split("_")[-1] for col in df.columns]
    maturity_dates = []

    for tenor in tenor_labels:
        if tenor.endswith("Y"):
            n = int(tenor[:-1])
            maturity_dates.append(REPORT_DATE + pd.DateOffset(years=n))
        elif tenor.endswith("M"):
            n = int(tenor[:-1])
            maturity_dates.append(REPORT_DATE + pd.DateOffset(months=n))
        else:
            raise ValueError(f"Unsupported tenor: {tenor}")

    start = np.array([REPORT_DATE] * len(maturity_dates), dtype="datetime64[D]")
    end = np.array(maturity_dates, dtype="datetime64[D]")
    maturities = daycount.yearfrac(start, end)

    # Lấy zero_rates từ ngày gần nhất <= REPORT_DATE
    valid_idx = df.index[df.index <= REPORT_DATE]
    report_idx = valid_idx.max()
    zero_rates = df.loc[report_idx].to_numpy(dtype=float)

    curves[group] = YieldCurve.from_zero_rates(
        maturities=maturities,
        zero_rates=zero_rates,
    )

    # Nếu ref curve riêng thì gán ở đây.
    # Nếu không có, dùng cùng curve group đó.
    ref_curves[group] = curves[group]

# -----------------------------
# 3) Đọc bond_df
# -----------------------------
bond_df = pd.read_csv(BOND_DATA_PATH)
display(bond_df)

# -----------------------------
# 4) Hàm chuyển continuous zero-rate -> simple rate
# -----------------------------
def continuous_to_simple_rate(curve: YieldCurve, tenor: float) -> float:
    """
    Từ zero rate liên tục R(0, T), chuyển sang lãi đơn:
        L = (exp(R*T) - 1) / T
    """
    R = curve.zero_rate(tenor)
    return float(np.expm1(R * tenor) / tenor)

# -----------------------------
# 5) Coupon maps
# -----------------------------
coupon_map_fixed = {
    1: CouponDef(accrual=0.25, fixed_rate=0.04),
    2: CouponDef(accrual=0.25, fixed_rate=0.04),
    3: CouponDef(accrual=0.25, fixed_rate=0.04),
    4: CouponDef(accrual=0.25, fixed_rate=0.04),
}

coupon_map_float = {
    1: CouponDef(accrual=0.25, fixed_rate=None, margin=0.0025, ref_tenor=0.25),
    2: CouponDef(accrual=0.25, fixed_rate=None, margin=0.0025, ref_tenor=0.25),
    3: CouponDef(accrual=0.25, fixed_rate=None, margin=0.0025, ref_tenor=0.50),
    4: CouponDef(accrual=0.25, fixed_rate=None, margin=0.0025, ref_tenor=0.50),
}

# -----------------------------
# 6) Định giá từng bond
# -----------------------------
results = []

for _, row in bond_df.iterrows():
    bond_id = row["bond_id"]

    # Chọn group theo bond_df
    if "group" in row and pd.notna(row["group"]):
        group = row["group"]
    else:
        group = bond_group_map[bond_id]

    # Chọn ref_curve theo bond_df nếu có cột ref_curve
    if "ref_curve" in row and pd.notna(row["ref_curve"]):
        ref_group = row["ref_curve"]
    else:
        ref_group = ref_curve_map.get(bond_id, group)

    if group not in curves:
        raise ValueError(f"Group {group} not found in curves.")
    if ref_group not in ref_curves:
        raise ValueError(f"Ref curve {ref_group} not found in ref_curves.")

    # ----------------------------------------------------
    # Tính n_steps / dt theo logic tree:
    # n_steps phải >= max(step trong coupon / exercise / maturity)
    # ----------------------------------------------------
    maturity_step = int(row["maturity_step"])
    exercise_step = int(row["exercise_step"]) if pd.notna(row["exercise_step"]) else maturity_step

    coupon_step_keys = [1, 2, 3, 4]  # ví dụ mẫu
    exercise_steps = [exercise_step]

    # Nếu style = Bermudan thì exercise ở nhiều bước
    if row["style"] == "Bermudan":
        exercise_steps = [2, 4]

    # Chọn dt theo schedule chung của curve:
    # Với dữ liệu này, bạn đang dùng cấu trúc step-based, nên dt=1.0 là hợp lệ với bond mẫu.
    dt = 1.0

    n_steps = max(
        maturity_step,
        max(coupon_step_keys),
        max(exercise_steps),
    )

    tree = MultiCurveHWTree(
        a_r=0.03,
        sigma_r=0.015,
        a_L=0.02,
        sigma_L=0.018,
        rho=0.20,
        disc_curve=curves[group],
        ref_curve=ref_curves[ref_group],
        dt=dt,
        n_steps=n_steps,
    )

    # ----------------------------------------------------
    # 6.1) Coupon cố định
    # ----------------------------------------------------
    if row["style"] == "European":
        coupon_map = coupon_map_fixed
        bond = BondSpec(
            face=float(row["face"]),
            maturity_step=maturity_step,
            coupons=coupon_map,
        )
        exercise = ExerciseSpec(
            call={exercise_step: float(row["strike"])}
        )

        full_price = tree.price(bond, exercise)
        straight_price = tree.price(bond)

    # ----------------------------------------------------
    # 6.2) Coupon thả nổi: L + margin
    # ----------------------------------------------------
    elif row["style"] == "Bermudan":
        # Nếu bond_df có cột ref_curve, chọn ref_curve theo bond đó.
        # Ref curve được dùng trong reference_rate(i, tenor):
        # ref_rate = (exp(R*tenor)-1)/tenor
        ref_curve = ref_curves[ref_group]

        # Với coupon thả nổi, BondSpec không nhận coupon_rate trực tiếp.
        # Nó lấy ref_curve ở thời điểm thanh toán coupon, cộng margin.
        # Vì vậy ta dùng CouponDef(... fixed_rate=None, margin=..., ref_tenor=...)
        coupon_map = coupon_map_float
        bond = BondSpec(
            face=float(row["face"]),
            maturity_step=maturity_step,
            coupons=coupon_map,
        )
        exercise = ExerciseSpec(
            put={2: 99.0, 4: 99.0}
        )

        full_price = tree.price(bond, exercise)
        straight_price = tree.price(bond)

    elif row["style"] == "American":
        # American = exercise trên mọi step trong [1, maturity_step]
        coupon_map = coupon_map_fixed
        bond = BondSpec(
            face=float(row["face"]),
            maturity_step=maturity_step,
            coupons=coupon_map,
        )
        exercise = ExerciseSpec(
            call={k: float(row["strike"]) for k in range(1, maturity_step + 1)}
        )

        full_price = tree.price(bond, exercise)
        straight_price = tree.price(bond)

    else:
        raise ValueError(f"Unsupported style: {row['style']}")

    results.append({
        "bond_id": bond_id,
        "group": group,
        "ref_group": ref_group,
        "style": row["style"],
        "maturity_step": maturity_step,
        "n_steps": n_steps,
        "dt": dt,
        "full_price": full_price,
        "straight_price": straight_price,
    })

results_df = pd.DataFrame(results)
display(results_df)

In [14]:
from pathlib import Path
import numpy as np
import pandas as pd
import sys

sys.path.insert(0, str(Path.cwd().parents[1]))

from src.curve import YieldCurve
from src.daycount import DayCount
from src.multi_hw_tree import BondSpec, CouponDef, ExerciseSpec, MultiCurveHWTree

# ============================================================
# 1) Paths, dates, and conventions
# ============================================================
repo_candidates = [
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path.cwd().resolve().parent.parent,
]

root = None
raw_dir = None
for candidate in repo_candidates:
    for raw_candidate in [candidate / "data" / "raw", candidate / "datasets" / "raw"]:
        bond_path = raw_candidate / "bonds_placeholder.csv"
        curve_path = raw_candidate / "Histocopy_FI_ZYC_VND_GD2_family.xlsx"
        exercise_path = raw_candidate / "bond_exercise_dates.csv"
        if bond_path.exists() and curve_path.exists() and exercise_path.exists():
            root = candidate
            raw_dir = raw_candidate
            break
    if root is not None:
        break

if root is None or raw_dir is None:
    raise FileNotFoundError("Unable to locate repository root with raw data files.")

BOND_DATA_PATH = raw_dir / "bonds_placeholder.csv"
CURVE_DATA_PATH = raw_dir / "Histocopy_FI_ZYC_VND_GD2_family.xlsx"
EXERCISE_MAP_PATH = raw_dir / "bond_exercise_dates.csv"

START_DATE = pd.to_datetime("2024-06-24")
END_DATE = pd.to_datetime("2026-07-30")
REPORT_DATE = pd.to_datetime("2026-06-03")

DISC_CONVENTION = "actactisda"
COUP_CONVENTION = "act365"

print(f"Using repo root: {root}")

# ============================================================
# 2) Load bond table and exercise-date map
# ============================================================
bond_df = pd.read_csv(BOND_DATA_PATH)
bond_df["issue_date"] = pd.to_datetime(bond_df["issue_date"])
bond_df["maturity_date"] = pd.to_datetime(bond_df["maturity_date"])

exercise_map = pd.read_csv(EXERCISE_MAP_PATH)
exercise_map["bond_id"] = exercise_map["bond_id"].astype(str)
exercise_map["exercise_dates"] = exercise_map["exercise_dates"].fillna("").astype(str)
exercise_map["exercise_dates"] = exercise_map["exercise_dates"].str.split(";")

exercise_dates_by_bond = {
    row["bond_id"]: [
        pd.to_datetime(d)
        for d in row["exercise_dates"]
        if str(d).strip() != ""
    ]
    for _, row in exercise_map.iterrows()
}


def get_exercise_dates_for_bond(bond_id: str):
    if bond_id in exercise_dates_by_bond:
        return exercise_dates_by_bond[bond_id]
    return []

# ============================================================
# 3) Build per-group curve dictionary
# ============================================================
groups = ["LB_G1", "LB_G2", "LB_G3", "LB_G4", "NBFI", "FB"]

dfs = {}
curves = {}
ref_curves = {}

daycount = DayCount.get(DISC_CONVENTION)

for group in groups:
    df = pd.read_excel(
        CURVE_DATA_PATH,
        sheet_name=f"FI ZYC VND_GD2_{group}",
        index_col=0,
    )
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    df = df.reindex(pd.date_range(start=START_DATE, end=END_DATE))
    dfs[group] = df

    tenor_labels = [col.split("_")[-1] for col in df.columns]
    maturity_dates = []

    for tenor in tenor_labels:
        if tenor.endswith("Y"):
            n = int(tenor[:-1])
            maturity_dates.append(REPORT_DATE + pd.DateOffset(years=n))
        elif tenor.endswith("M"):
            n = int(tenor[:-1])
            maturity_dates.append(REPORT_DATE + pd.DateOffset(months=n))
        else:
            raise ValueError(f"Unsupported tenor: {tenor}")

    start = np.array([REPORT_DATE] * len(maturity_dates), dtype="datetime64[D]")
    end = np.array(maturity_dates, dtype="datetime64[D]")
    maturities = daycount.yearfrac(start, end)

    valid_idx = df.index[df.index <= REPORT_DATE]
    report_idx = valid_idx.max()
    zero_rates = df.loc[report_idx].to_numpy(dtype=float)

    curves[group] = YieldCurve.from_zero_rates(
        maturities=maturities,
        zero_rates=zero_rates,
    )
    ref_curves[group] = curves[group]

# ============================================================
# 4) Helper: infer a step-based tree grid from the bond.
# ============================================================
def infer_tree_grid_for_bond(row: pd.Series) -> tuple[float, int]:
    """
    For this tree implementation, the bond schedule must be expressed in
    discrete time steps. The notebook data already carries a step-based
    maturity and exercise pattern, so we map the bond directly to
    dt = 1.0 and the required number of steps.
    """
    maturity_step = int(row["maturity_step"])
    exercise_step = (
        int(row["exercise_step"])
        if pd.notna(row["exercise_step"])
        else maturity_step
    )
    exercise_steps = [exercise_step]

    exercise_steps_raw = str(row["exercise_steps"]).strip()
    if exercise_steps_raw and exercise_steps_raw.lower() != "nan":
        exercise_steps.extend(
            int(s)
            for s in exercise_steps_raw.split(";")
            if str(s).strip() not in {"", "nan", "NaN"}
        )

    n_steps = max(maturity_step, max(exercise_steps))
    return 1.0, n_steps

# ============================================================
# 5) Prepare coupon maps
# ============================================================
coupon_map_fixed = {
    1: CouponDef(accrual=0.25, fixed_rate=0.04),
    2: CouponDef(accrual=0.25, fixed_rate=0.04),
    3: CouponDef(accrual=0.25, fixed_rate=0.04),
    4: CouponDef(accrual=0.25, fixed_rate=0.04),
}

coupon_map_float = {
    1: CouponDef(accrual=0.25, fixed_rate=None, margin=0.0025, ref_tenor=0.25),
    2: CouponDef(accrual=0.25, fixed_rate=None, margin=0.0025, ref_tenor=0.25),
    3: CouponDef(accrual=0.25, fixed_rate=None, margin=0.0025, ref_tenor=0.50),
    4: CouponDef(accrual=0.25, fixed_rate=None, margin=0.0025, ref_tenor=0.50),
}

# ============================================================
# 6) Price bonds one by one
# ============================================================
results = []

for _, row in bond_df.iterrows():
    bond_id = str(row["bond_id"])
    style = str(row["style"]).strip().lower()
    group = str(row["group"]).strip() if "group" in row and pd.notna(row["group"]) else None

    if group is None:
        group = {
            "BOND_EUR_01": "LB_G1",
            "BOND_BERM_02": "FB",
        }.get(bond_id, "LB_G1")

    ref_group = (
        str(row["ref_curve"]).strip()
        if "ref_curve" in row and pd.notna(row["ref_curve"])
        else group
    )

    if group not in curves:
        raise ValueError(f"Group {group} not found in curves.")
    if ref_group not in ref_curves:
        raise ValueError(f"Ref curve {ref_group} not found in ref_curves.")

    dt, n_steps = infer_tree_grid_for_bond(row)

    tree = MultiCurveHWTree(
        a_r=0.03,
        sigma_r=0.015,
        a_L=0.02,
        sigma_L=0.018,
        rho=0.20,
        disc_curve=curves[group],
        ref_curve=ref_curves[ref_group],
        dt=dt,
        n_steps=n_steps,
    )

    maturity_step = int(row["maturity_step"])
    face = float(row["face"])

    if style == "european":
        coupon_map = coupon_map_fixed
        bond = BondSpec(
            face=face,
            maturity_step=maturity_step,
            coupons=coupon_map,
        )
        exercise = ExerciseSpec(
            call={int(row["exercise_step"]): float(row["strike"])}
        )
        full_price = tree.price(bond, exercise)
        straight_price = tree.price(bond)

    elif style == "bermudan":
        coupon_map = coupon_map_float
        bond = BondSpec(
            face=face,
            maturity_step=maturity_step,
            coupons=coupon_map,
        )

        exercise_steps = [
            int(s)
            for s in str(row["exercise_steps"]).split(";")
            if str(s).strip() != ""
        ]
        if not exercise_steps:
            exercise_steps = [2, 4]

        exercise = ExerciseSpec(
            put={step: float(row["strike"]) for step in exercise_steps}
        )
        full_price = tree.price(bond, exercise)
        straight_price = tree.price(bond)

    elif style == "american":
        coupon_map = coupon_map_fixed
        bond = BondSpec(
            face=face,
            maturity_step=maturity_step,
            coupons=coupon_map,
        )
        exercise = ExerciseSpec(
            call={step: float(row["strike"]) for step in range(1, maturity_step + 1)}
        )
        full_price = tree.price(bond, exercise)
        straight_price = tree.price(bond)

    else:
        raise ValueError(f"Unsupported style '{style}' for bond {bond_id}")

    results.append({
        "bond_id": bond_id,
        "group": group,
        "ref_group": ref_group,
        "style": style,
        "maturity_step": maturity_step,
        "dt": dt,
        "n_steps": n_steps,
        "full_price": full_price,
        "straight_price": straight_price,
    })

results_df = pd.DataFrame(results)
display(results_df)

Using repo root: E:\VCB\202607\vcb-snh


,bond_id,group,ref_group,style,maturity_step,dt,n_steps,full_price,straight_price
0,BOND_EUR_01,LB_G1,LB_G1,european,4,1.0,4,75.480859,75.480859
1,BOND_BERM_02,FB,FB,bermudan,4,1.0,4,86.449352,77.519169


In [20]:
'''<VSCode.Cell language="markdown">
# End-to-end pricing pipeline for per-group curves + per-bond option pricing

This notebook:
- loads curve data per group from the workbook
- constructs a discount curve and reference curve for each group
- reads the bond table from the raw CSV
- maps exercise dates per bond from a CSV lookup
- computes `dt` and `n_steps` per bond so coupon dates, exercise dates, and maturity land on tree nodes
- prices each bond with European / Bermudan / American style
- handles fixed coupons and floating coupons using `ref_curve`
</VSCode.Cell>

<VSCode.Cell language="python">
'''
from pathlib import Path
from math import gcd
import numpy as np
import pandas as pd
import sys

sys.path.insert(0, str(Path.cwd().parents[1]))

from src.curve import YieldCurve
from src.daycount import DayCount
from src.multi_hw_tree import BondSpec, CouponDef, ExerciseSpec, MultiCurveHWTree

# ============================================================
# 1) Paths, dates, and conventions
# ============================================================
root = Path.cwd().resolve().parent.parent

BOND_DATA_PATH = root / "datasets" / "raw" / "bonds_placeholder.csv"
CURVE_DATA_PATH = root / "datasets" / "raw" / "Histocopy_FI_ZYC_VND_GD2_family.xlsx"
EXERCISE_MAP_PATH = root / "datasets" / "raw" / "bond_exercise_dates.csv"

START_DATE = pd.to_datetime("2024-06-24")
END_DATE = pd.to_datetime("2026-07-30")
REPORT_DATE = pd.to_datetime("2026-06-03")

DISC_CONVENTION = "actactisda"
COUP_CONVENTION = "act365"

# ============================================================
# 2) Load bond table and exercise-date map
# ============================================================
bond_df = pd.read_csv(BOND_DATA_PATH)
bond_df["issue_date"] = pd.to_datetime(bond_df["issue_date"])
bond_df["maturity_date"] = pd.to_datetime(bond_df["maturity_date"])

exercise_map = pd.read_csv(EXERCISE_MAP_PATH)
exercise_map["bond_id"] = exercise_map["bond_id"].astype(str)
exercise_map["exercise_dates"] = exercise_map["exercise_dates"].fillna("").astype(str)
exercise_map["exercise_dates"] = exercise_map["exercise_dates"].str.split(";")

# Convert mapping to dict {bond_id: [exercise_dates]}
exercise_dates_by_bond = {
    row["bond_id"]: [pd.to_datetime(d) for d in row["exercise_dates"] if str(d).strip() != ""]
    for _, row in exercise_map.iterrows()
}

# Fallback if exercise date file is not present
def get_exercise_dates_for_bond(bond_id: str):
    if bond_id in exercise_dates_by_bond:
        return exercise_dates_by_bond[bond_id]
    return []

# ============================================================
# 3) Build per-group curve dictionary
# ============================================================
groups = ["LB_G1", "LB_G2", "LB_G3", "LB_G4", "NBFI", "FB"]

dfs = {}
curves = {}
ref_curves = {}

daycount = DayCount.get(DISC_CONVENTION)

for group in groups:
    df = pd.read_excel(
        CURVE_DATA_PATH,
        sheet_name=f"FI ZYC VND_GD2_{group}",
        index_col=0,
    )
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    df = df.reindex(pd.date_range(start=START_DATE, end=END_DATE))
    dfs[group] = df

    # infer maturities from tenor labels, e.g. 3M, 6M, 1Y, 2Y, ...
    tenor_labels = [col.split("_")[-1] for col in df.columns]
    maturity_dates = []

    for tenor in tenor_labels:
        if tenor.endswith("Y"):
            n = int(tenor[:-1])
            maturity_dates.append(REPORT_DATE + pd.DateOffset(years=n))
        elif tenor.endswith("M"):
            n = int(tenor[:-1])
            maturity_dates.append(REPORT_DATE + pd.DateOffset(months=n))
        else:
            raise ValueError(f"Unsupported tenor: {tenor}")

    start = np.array([REPORT_DATE] * len(maturity_dates), dtype="datetime64[D]")
    end = np.array(maturity_dates, dtype="datetime64[D]")
    maturities = daycount.yearfrac(start, end)

    # pick the last valid rate point on or before REPORT_DATE
    valid_idx = df.index[df.index <= REPORT_DATE]
    report_idx = valid_idx.max()
    zero_rates = df.loc[report_idx].to_numpy(dtype=float)

    curves[group] = YieldCurve.from_zero_rates(
        maturities=maturities,
        zero_rates=zero_rates,
    )

    # if ref_curve is grouped separately, map it to the same curve for now
    ref_curves[group] = curves[group]

# ============================================================
# 4) Helper: continuous zero rate -> simple rate
# ============================================================
def continuous_to_simple_rate(curve: YieldCurve, tenor: float) -> float:
    """
    Convert continuously-compounded zero rate R to simple rate L:
        L = (exp(R * tenor) - 1) / tenor
    """
    R = curve.zero_rate(tenor)
    return float(np.expm1(R * tenor) / tenor)

# ============================================================
# 5) Per-bond schedule helper: choose dt and n_steps
# ============================================================
def infer_tree_grid_for_bond_old(row: pd.Series) -> tuple[float, int]:
    """
    Find dt and n_steps such that:
      - coupon dates
      - exercise dates
      - maturity date
    all fall on the tree grid.
    """
    issue_date = pd.to_datetime(row["issue_date"])
    maturity_date = pd.to_datetime(row["maturity_date"])

    # coupon dates by step: use accrual to derive quarterly/semiannual schedule
    coupon_accrual = float(row["coupon_accrual"])
    maturity_step = int(row["maturity_step"])

    # coupon dates from the issue date forward
    coupon_dates = [
        issue_date + pd.DateOffset(days=int(round(365 * coupon_accrual * k)))
        for k in range(1, maturity_step + 1)
    ]

    # exercise dates from bond-specific csv mapping
    bond_id = str(row["bond_id"])
    exercise_dates = get_exercise_dates_for_bond(bond_id)
    exercise_dates = sorted(set(exercise_dates))

    # all event dates
    event_dates = sorted(set(coupon_dates + exercise_dates + [maturity_date]))

    # year fractions from issue date to each event following the repo daycount
    daycount_obj = DayCount.get(DISC_CONVENTION)
    times = daycount_obj.yearfrac(
        np.array([issue_date] * len(event_dates), dtype="datetime64[D]"),
        np.array(event_dates, dtype="datetime64[D]"),
    )

    # choose a common dt as the smallest positive step between dates in time-space
    t_sorted = np.sort(times)
    deltas = np.diff(t_sorted)
    positive_deltas = deltas[deltas > 0]
    if positive_deltas.size == 0:
        raise ValueError("No positive step found in schedule.")

    dt = float(np.min(positive_deltas))

    # Ensure dates land on grid nodes: round-and-check
    grid_times = np.round(t_sorted / dt) * dt
    if not np.allclose(t_sorted, grid_times, atol=1e-9, rtol=0.0):
        raise ValueError(
            f"Bond {bond_id} schedule is not aligned to a common dt={dt} grid."
        )

    n_steps = int(round(t_sorted[-1] / dt))
    return dt, n_steps



def infer_tree_grid_for_bond(row: pd.Series, daycount_convention: str = "act365"):
    """
    Từ issue_date, coupon_dates, exercise_dates, maturity_date:
      1) tính khoảng cách ngày giữa các event liên tiếp
      2) lấy gcd của các khoảng cách ngày
      3) đặt dt = gcd_days / 365
      4) kiểm tra các yearfraction event có rơi đúng lên node không
    """
    bond_id = str(row["bond_id"])
    issue_date = pd.to_datetime(row["issue_date"])
    maturity_date = pd.to_datetime(row["maturity_date"])

    # coupon dates theo coupon_accrual
    coupon_accrual = float(row["coupon_accrual"])
    maturity_step = int(row["maturity_step"])
    coupon_dates = [
        issue_date + pd.DateOffset(days=int(round(365 * coupon_accrual * k)))
        for k in range(1, maturity_step + 1)
    ]

    # exercise dates from lookup
    exercise_dates = get_exercise_dates_for_bond(bond_id)
    exercise_dates = sorted(set(exercise_dates))

    # tạo event list
    event_dates = sorted(set(coupon_dates + exercise_dates + [maturity_date]))

    # 1) khoảng cách ngày giữa các event liên tiếp
    day_gaps = np.diff(
        np.array([issue_date] + event_dates, dtype="datetime64[D]")
    ).astype(int)

    # 2) gcd của các khoảng cách ngày
    x_days = int(day_gaps[0])
    for d in day_gaps[1:]:
        x_days = gcd(x_days, int(d))

    # 3) dt theo năm
    dt = x_days / 365.0

    # 4) year fraction theo daycount thực tế
    daycount_obj = DayCount.get(daycount_convention)
    times = daycount_obj.yearfrac(
        np.array([issue_date] * len(event_dates), dtype="datetime64[D]"),
        np.array(event_dates, dtype="datetime64[D]"),
    )
    t_sorted = np.sort(times)

    # 5) ép các time về grid
    grid_times = np.round(t_sorted / dt) * dt

    # 6) nếu lệch thì báo lỗi
    if not np.allclose(t_sorted, grid_times, atol=1e-9, rtol=0.0):
        raise ValueError(
            f"Bond {bond_id} cannot be aligned to a common dt={dt} grid "
            f"with gcd-days={x_days}."
        )

    # n_steps là số bước tới maturity trên grid đó
    n_steps = int(round(t_sorted[-1] / dt))

    return dt, n_steps
# ============================================================
# 6) Prepare coupon maps
# ============================================================
coupon_schedule_df = pd.DataFrame([
    {"bond_id": "BOND_EUR_01", "step": 1, "accrual": 0.25, "coupon_type": "float", "margin": 0.0025, "ref_tenor": 0.25},
    {"bond_id": "BOND_EUR_01", "step": 2, "accrual": 0.25, "coupon_type": "float", "margin": 0.0025, "ref_tenor": 0.25},
    {"bond_id": "BOND_EUR_01", "step": 3, "accrual": 0.25, "coupon_type": "float", "margin": 0.0030, "ref_tenor": 0.50},
    {"bond_id": "BOND_EUR_01", "step": 4, "accrual": 0.25, "coupon_type": "float", "margin": 0.0030, "ref_tenor": 0.50},
])

def build_coupon_map_from_schedule(row: pd.Series, coupon_schedule_df: pd.DataFrame):
    bond_id = str(row["bond_id"])
    sub = coupon_schedule_df[coupon_schedule_df["bond_id"] == bond_id].sort_values("step")

    coupon_map = {}

    for _, s in sub.iterrows():
        step = int(s["step"])
        accrual = float(s["accrual"])  # vd 0.25, 0.5, 1.0
        coupon_type = str(s["coupon_type"]).strip().lower()

        if coupon_type == "fixed":
            coupon_map[step] = CouponDef(
                accrual=accrual,
                fixed_rate=float(s["fixed_rate"]),
            )
        elif coupon_type == "float":
            coupon_map[step] = CouponDef(
                accrual=accrual,
                fixed_rate=None,
                margin=float(s["margin"]),
                ref_tenor=float(s["ref_tenor"]),
                floor=s["floor"] if pd.notna(s["floor"]) else None,
                cap=s["cap"] if pd.notna(s["cap"]) else None,
            )
        else:
            raise ValueError(f"Unsupported coupon_type={coupon_type}")

    return coupon_map

coupon_map_fixed = {
    1: CouponDef(accrual=0.25, fixed_rate=0.04),
    2: CouponDef(accrual=0.25, fixed_rate=0.04),
    3: CouponDef(accrual=0.25, fixed_rate=0.04),
    4: CouponDef(accrual=0.25, fixed_rate=0.04),
}

coupon_map_float = {
    1: CouponDef(accrual=0.25, fixed_rate=None, margin=0.0025, ref_tenor=0.25),
    2: CouponDef(accrual=0.25, fixed_rate=None, margin=0.0025, ref_tenor=0.25),
    3: CouponDef(accrual=0.25, fixed_rate=None, margin=0.0025, ref_tenor=0.50),
    4: CouponDef(accrual=0.25, fixed_rate=None, margin=0.0025, ref_tenor=0.50),
}

# ============================================================
# 7) Price bonds one by one
# ============================================================
results = []

for _, row in bond_df.iterrows():
    bond_id = str(row["bond_id"])
    style = str(row["style"]).strip().lower()
    group = str(row["group"]).strip() if "group" in row and pd.notna(row["group"]) else None

    # determine the group for this bond
    if group is None:
        group = {
            "BOND_EUR_01": "LB_G1",
            "BOND_BERM_02": "FB",
        }.get(bond_id, "LB_G1")

    # determine the ref curve group
    ref_group = str(row["ref_curve"]).strip() if "ref_curve" in row and pd.notna(row["ref_curve"]) else group

    if group not in curves:
        raise ValueError(f"Group {group} not found in curves.")
    if ref_group not in ref_curves:
        raise ValueError(f"Ref curve {ref_group} not found in ref_curves.")

    # infer dt and n_steps for this bond from its schedule
    dt, n_steps = infer_tree_grid_for_bond(row)
    print(dt)

    tree = MultiCurveHWTree(
        a_r=0.03,
        sigma_r=0.015,
        a_L=0.02,
        sigma_L=0.018,
        rho=0.20,
        disc_curve=curves[group],
        ref_curve=ref_curves[ref_group],
        dt=dt,
        n_steps=n_steps,
    )

    maturity_step = int(row["maturity_step"])
    face = float(row["face"])
    coupon_accrual = float(row["coupon_accrual"])
    coupon_map = build_coupon_map_from_schedule(row, coupon_schedule_df)
    bond = BondSpec(
        face=face,
        maturity_step=maturity_step,
        coupons=coupon_map,
    )
    # --------------------------------------------------------
    # European: one exercise event
    # --------------------------------------------------------
    if style == "european":
        # coupon_map = coupon_map_fixed
    
        exercise = ExerciseSpec(
            call={int(row["exercise_step"]): float(row["strike"])}
        )

        full_price = tree.price(bond, exercise)
        straight_price = tree.price(bond)

    # --------------------------------------------------------
    # Bermudan: multiple exercise dates in exercise map
    # --------------------------------------------------------
    elif style == "bermudan":
        # Floating coupon: use the ref curve to derive the reference rate
        # The tree consumes floating coupons through CouponDef(... fixed_rate=None, margin=..., ref_tenor=...)

        exercise_steps = [int(s) for s in str(row["exercise_steps"]).split(";") if str(s).strip() != ""]
        if not exercise_steps:
            exercise_steps = [2, 4]

        exercise = ExerciseSpec(
            put={step: float(row["strike"]) for step in exercise_steps}
        )

        full_price = tree.price(bond, exercise)
        straight_price = tree.price(bond)

    # --------------------------------------------------------
    # American: exercise allowed at all steps in the window
    # --------------------------------------------------------
    elif style == "american":

        exercise = ExerciseSpec(
            call={step: float(row["strike"]) for step in range(1, maturity_step + 1)}
        )

        full_price = tree.price(bond, exercise)
        straight_price = tree.price(bond)

    else:
        raise ValueError(f"Unsupported style '{style}' for bond {bond_id}")

    results.append({
        "bond_id": bond_id,
        "group": group,
        "ref_group": ref_group,
        "style": style,
        "maturity_step": maturity_step,
        "dt": dt,
        "n_steps": n_steps,
        "full_price": full_price,
        "straight_price": straight_price,
    })

results_df = pd.DataFrame(results)
display(results_df)

0.0027397260273972603


KeyError: 'floor'